In [ ]:
import numpy as np
import pandas as pd
import os

# ==========================================
# 🟦 MODÜL 1: data_generator.py (Simülasyon)
# ==========================================
def generate_raw_data(n_records=500):
    """
    NumPy kullanarak gerçekçi, bozuk ve eksik veriler üretir.
    """
    np.random.seed(42)

    cities = ['Istanbul', 'Ankara', 'Izmir', 'Bursa', 'Antalya', 'Diyarbakir']
    products = ['Laptop', 'Smartphone', 'Tablet', 'Monitor', 'Keyboard']
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']

    data = {
        'order_id': np.arange(1000, 1000 + n_records),
        'customer_id': np.random.randint(5000, 6000, size=n_records),
        'city': np.random.choice(cities, size=n_records),
        'product': np.random.choice(products, size=n_records),
        'sales_amount': np.random.uniform(100, 15000, size=n_records),
        'month': np.random.choice(months, size=n_records),
        'quantity': np.random.randint(1, 10, size=n_records)
    }

    df = pd.DataFrame(data)

    # ❗ Bilerek Bozuk Veri Ekleme (NaN ve Outliers)
    # Satış miktarının %10'unu NaN yap
    nan_indices = np.random.choice(df.index, size=int(n_records * 0.1), replace=False)
    df.loc[nan_indices, 'sales_amount'] = np.nan

    # Şehir isimlerini kirlet (Büyük/Küçük harf karmaşası)
    df['city'] = df['city'].apply(lambda x: x.upper() if np.random.rand() > 0.8 else x.lower())

    # Uç değerler (Outliers)
    outlier_indices = np.random.choice(df.index, size=5, replace=False)
    df.loc[outlier_indices, 'sales_amount'] = df.loc[outlier_indices, 'sales_amount'] * 50

    return df

# ==========================================
# 🟦 MODÜL 2: preprocessing.py (Temizleme)
# ==========================================
def preprocess_data(df):
    """
    Pandas ve Python Core kullanarak veriyi temizler ve normalize eder.
    """
    print("... Veri ön işleme başlatıldı ...")
    df_clean = df.copy()

    try:
        # 1. Eksik Veri Yönetimi
        # Satış tutarı boş olanları medyan ile doldur (Outlier etkisini azaltmak için)
        median_sales = df_clean['sales_amount'].median()
        df_clean['sales_amount'] = df_clean['sales_amount'].fillna(median_sales)

        # 2. String Normalizasyonu
        df_clean['city'] = df_clean['city'].str.capitalize()
        df_clean['product'] = df_clean['product'].str.strip()

        # 3. Yeni Sütun Üretimi (Feature Engineering)
        kdv_rate = 0.20
        df_clean['tax_amount'] = df_clean['sales_amount'] * kdv_rate
        df_clean['total_with_tax'] = df_clean['sales_amount'] + df_clean['tax_amount']

        # Birim fiyat hesapla
        df_clean['unit_price'] = df_clean['sales_amount'] / df_clean['quantity']

        # 4. Outlier Temizliği (Basit IQR Yöntemi)
        Q1 = df_clean['sales_amount'].quantile(0.25)
        Q3 = df_clean['sales_amount'].quantile(0.75)
        IQR = Q3 - Q1
        upper_limit = Q3 + 1.5 * IQR
        # Uç değerleri üst sınırla baskıla
        df_clean.loc[df_clean['sales_amount'] > upper_limit, 'sales_amount'] = upper_limit

        return df_clean

    except Exception as e:
        print(f"HATA: Preprocessing aşamasında bir sorun oluştu: {e}")
        return None

# ==========================================
# 🟦 MODÜL 3: analytics.py (Analiz Motoru)
# ==========================================
def perform_analytics(df):
    """
    NumPy vektörel işlemler ve Pandas GroupBy ile derin analiz yapar.
    """
    # Şehir bazlı performans
    city_perf = df.groupby('city')['sales_amount'].agg(['sum', 'mean', 'count']).sort_values(by='sum', ascending=False)

    # Ürün bazlı trend
    product_perf = df.groupby('product')['sales_amount'].sum().to_dict()

    # Ay bazlı trend
    month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
    df['month'] = pd.Categorical(df['month'], categories=month_order, ordered=True)
    monthly_trend = df.groupby('month', observed=False)['sales_amount'].sum()

    # NumPy ile Performans Skoru (Vektörel)
    # Satışları 0-1 arasına normalize et
    sales_array = df['sales_amount'].values
    normalized_scores = (sales_array - np.min(sales_array)) / (np.max(sales_array) - np.min(sales_array))
    df['performance_score'] = normalized_scores

    return {
        'city_metrics': city_perf,
        'product_metrics': product_perf,
        'monthly_trend': monthly_trend,
        'overall_total': np.sum(sales_array)
    }

# ==========================================
# 🟦 MODÜL 4: reporting.py (Karar Destek)
# ==========================================
def generate_executive_report(analytics_results):
    """
    Yöneticiye özet rapor ve aksiyon önerileri sunar.
    """
    metrics = analytics_results['city_metrics']

    top_city = metrics.index[0]
    worst_city = metrics.index[-1]

    best_month = analytics_results['monthly_trend'].idxmax()

    # Riskli bölgeler (Ortalama satışı genel ortalamanın altında kalanlar)
    avg_sales = metrics['mean'].mean()
    risky_regions = metrics[metrics['mean'] < avg_sales].index.tolist()

    report = f"""
    ╔══════════════════════════════════════════════════╗
    ║      ENTERPRISE SALES INTELLIGENCE REPORT        ║
    ╚══════════════════════════════════════════════════╝

    📍 GENEL DURUM:
    - Toplam Brüt Satış: {analytics_results['overall_total']:,.2f} TL
    - En Başarılı Şehir: {top_city}
    - En Zayıf Performans: {worst_city}
    - En Verimli Ay: {best_month}

    ⚠️ RİSK ANALİZİ:
    - Riskli Bölgeler (Düşük Ortalama): {', '.join(risky_regions)}

    💡 KARAR DESTEK ÖNERİSİ:
    - {top_city} bölgesindeki stokları %15 artırın.
    - {risky_regions[0] if risky_regions else 'Genel'} için yeni bir kampanya kurgulayın.
    """
    return report

# ==========================================
# 🟦 MODÜL 5: main.py (Sistem Akışı)
# ==========================================
if __name__ == "__main__":
    print("--- Enterprise Sales & Decision Support System Başlatılıyor ---\n")

    # 1. Veri Üretimi
    raw_data = generate_raw_data(1000)
    print(f"1. ADIM: {len(raw_data)} adet ham veri üretildi.")

    # 2. Ön İşleme
    clean_data = preprocess_data(raw_data)

    if clean_data is not None:
        # 3. Analiz
        results = perform_analytics(clean_data)
        print("3. ADIM: Analitik motoru verileri işledi.")

        # 4. Raporlama
        final_report = generate_executive_report(results)
        print(final_report)

        # 5. Excel Çıktısı
        try:
            output_file = "Sales_Intelligence_Report.xlsx"
            with pd.ExcelWriter(output_file) as writer:
                clean_data.to_excel(writer, sheet_name='CleanData', index=False)
                results['city_metrics'].to_excel(writer, sheet_name='CityAnalytics')
            print(f"✅ Başarılı: Detaylı rapor '{output_file}' olarak kaydedildi.")
        except Exception as e:
            print(f"Excel yazım hatası: {e}")

    print("\n--- Sistem Çalışması Tamamlandı ---")

--- Enterprise Sales & Decision Support System Başlatılıyor ---

1. ADIM: 1000 adet ham veri üretildi.
... Veri ön işleme başlatıldı ...
3. ADIM: Analitik motoru verileri işledi.

    ╔══════════════════════════════════════════════════╗
    ║      ENTERPRISE SALES INTELLIGENCE REPORT        ║
    ╚══════════════════════════════════════════════════╝
    
    📍 GENEL DURUM:
    - Toplam Brüt Satış: 7,689,665.45 TL
    - En Başarılı Şehir: Istanbul
    - En Zayıf Performans: Bursa
    - En Verimli Ay: Mar
    
    ⚠️ RİSK ANALİZİ:
    - Riskli Bölgeler (Düşük Ortalama): Antalya, Izmir, Ankara, Bursa
    
    💡 KARAR DESTEK ÖNERİSİ:
    - Istanbul bölgesindeki stokları %15 artırın.
    - Antalya için yeni bir kampanya kurgulayın.
    
✅ Başarılı: Detaylı rapor 'Sales_Intelligence_Report.xlsx' olarak kaydedildi.

--- Sistem Çalışması Tamamlandı ---
